# Deep Convolutional GAN (DCGAN)
Deep Learning (D6421) Sesi 07 — Studi Komparatif Model Generatif

Convolutional generator (ConvTranspose + BatchNorm + ReLU, Tanh out) and discriminator (strided Conv + LeakyReLU).

**Dataset:** Fashion-MNIST · **Split:** 80/10/10 · **Seed:** 42 · **Batch size:** 128

## 0. Setup

In [ ]:
import os, json, time, random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, ConcatDataset, random_split
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from tqdm.auto import tqdm

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device,
      "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "")

In [ ]:
MODEL_NAME = "dcgan"

DATA_DIR = "data"
CKPT_DIR = "checkpoints"
FIG_DIR = "figures"
RESULTS_DIR = "results"
for d in (DATA_DIR, CKPT_DIR, FIG_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)

BATCH_SIZE = 128
NOISE_DIM = 100

## 1. Dataset & Split (80 / 10 / 10)
The 60k train and 10k test sets of Fashion-MNIST are merged (70k images) and split
80/10/10 with a fixed generator (seed 42), so **every notebook gets exactly the same split**.
Images are in [0, 1]. Models that need [-1, 1] (GAN, DCGAN, DDPM) rescale with `x * 2 - 1`.

In [ ]:
transform = transforms.ToTensor()
full = ConcatDataset([
    datasets.FashionMNIST(DATA_DIR, train=True, download=True, transform=transform),
    datasets.FashionMNIST(DATA_DIR, train=False, download=True, transform=transform),
])
n = len(full)
n_train, n_val = int(0.8 * n), int(0.1 * n)
n_test = n - n_train - n_val
train_set, val_set, test_set = random_split(
    full, [n_train, n_val, n_test], generator=torch.Generator().manual_seed(SEED))

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

CLASSES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
           "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
print(f"train={len(train_set)}  val={len(val_set)}  test={len(test_set)}")

## 2. Shared Evaluation Utilities
Same code in every notebook so metrics are comparable: MSE / PSNR / SSIM on the test set,
FID for generated samples, and inference latency (ms per batch of 64).

## 3. Model Architecture

## 4. Training

## 5. Evaluation (Test Set)
- FID of samples from z ~ N(0, I)
- Training time per epoch
- Latency (ms / batch 64)
- Stability notes

## 6. Visualizations
- 4x4 random samples
- Generator vs discriminator loss curves
- Linear interpolation α ∈ {0, .2, .4, .6, .8, 1}

## 7. Save Results

In [ ]:
results = {"model": MODEL_NAME}
with open(os.path.join(RESULTS_DIR, f"{MODEL_NAME}.json"), "w") as f:
    json.dump(results, f, indent=2)